# Set 7: Hashes

This is **the first of two sets we generated after the original 6**.

Unlike the last few sets, **this set is a hodge-podge**.  It also includes some of the few challenges we have that probably aren't useful against real targets (they were fun enough to include anyways).  On the other hand, we also include a challenge that models the CRIME attack on TLS.

This set is **hard**.  There's a significant amount of programming, and Wang's attack in particular is as difficult as anything we've done.

- [Preliminaries](#Preliminaries)
- [Challenge 49: CBC-MAC message forgery](#Challenge-49:-CBC-MAC-message-forgery)
- [Challenge 50: Hashing with CBC-MAC](#Challenge-50:-Hashing-with-CBC-MAC)
- [Challenge 51: Compression ratio side-channel attacks](#Challenge-51:-Compression-ratio-side-channel-attacks)

## Preliminaries

In [1]:
from random import randbytes
from zlib import compress

# From pyca/cryptography
from cryptography.hazmat.primitives import padding
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

def aes_128_ctr_crypt(text, key, nonce):
    # Symmetric encryption/decryption
    encryptor = Cipher(algorithms.AES128(key), modes.CTR(nonce)).encryptor()
    return encryptor.update(text) + encryptor.finalize()

def pad_pkcs7(text):
    padder = padding.PKCS7(128).padder()
    return padder.update(text) + padder.finalize()

def aes_128_cbc_encrypt(ptext, key, iv):
    encryptor = Cipher(algorithms.AES128(key), modes.CBC(iv)).encryptor()
    return encryptor.update(pad_pkcs7(ptext)) + encryptor.finalize()

def xor(x, y):
    return bytes(xb^yb for xb, yb in zip(x, y))

zero = bytes(16)

## Challenge 49: CBC-MAC message forgery

Let's talk about CBC-MAC.

CBC-MAC is like this:

1. Take the plaintext P.
2. Encrypt P under CBC with key K, yielding ciphertext C.
3. Chuck all of C but the last block C\[n].
4. C\[n] is the MAC.

Suppose there's an online banking application, and it carries out user requests by talking to an API server over the network.  Each request looks like this:

```
message || IV || MAC
```

The message looks like this:

```
from=#{from_id}&to=#{to_id}&amount=#{amount}
```

Now, write an API server and a web frontend for it.  (NOTE: No need to get ambitious and write actual servers and web apps.  Totally fine to go lo-fi on this one.)  The client and server should share a secret key K to sign and verify messages.

The API server should accept messages, verify signatures, and carry out each transaction if the MAC is valid.  It's also publicly exposed - the attacker can submit messages freely assuming he can forge the right MAC.

The web client should allow the attacker to generate valid messages for accounts he controls.  (Feel free to sanitize params if you're feeling anal-retentive.)  Assume the attacker is in a position to capture and inspect messages from the client to the API server.

One thing we haven't discussed is the IV.  Assume the client generates a per-message IV and sends it along with the MAC.  That's how CBC works, right?

Wrong.

For messages signed under CBC-MAC, an attacker-controlled IV is a liability.  Why?  Because it yields full control over the first block of the message.

Use this fact to generate a message transferring 1M spacebucks from a target victim's account into your account.

I'll wait. Just let me know when you're done.

... waiting

... waiting

... waiting

All done?  Great - I knew you could do it!

---

This challenge and the next appear to have been inspired by Matthew Green's blog post, [Why I hate CBC-MAC](https://blog.cryptographyengineering.com/2013/02/15/why-i-hate-cbc-mac/).  The attack here is trivial to implement, but it seems of limited value since only the first block (16 bytes) of the message can be modified.  Still, the first block can be modified at will.  Here it is convenient to refer back to how CBC mode works:

![](cbc-mode.png)

Let $O$ be the first block of the original message, $I$ the initialization vector, and $O'$ the desired replacement block.  Since the first ciphertext block is computed by encrypting $I \oplus O$, we can just replace that with $I' \oplus O'$ where $I' = I \oplus O \oplus O'$ to recreate the original CBC-MAC computation.

But the requested forgery doesn't seem realistically possible.  Given the message syntax and a reasonable length for account IDs, the recipient account won't occur within the first 16 bytes.  And even if it does, how could one modify the amount, which occurs even later in the message?  Another solver suggested submitting a legitimate, innocuous request first, `from=attacker_id&to=attacker_id&amount=1000000`, and then modifying the sender, not the recipient.  But this assumes that the server would allow such requests, and that the attacker has sufficient funds to support it.

In [2]:
key = randbytes(16)

def cbc_mac(text, iv):
    return aes_128_cbc_encrypt(text, key, iv)[-16:]

def authenticate(text, iv, mac):
    return cbc_mac(text, iv) == mac

#                  |..............|  <-- first block
message =        b"from=Bob&to=Dave&amount=5"
forged_message = b"from=Bob&to=Greg&amount=5"

# The message and the following are intercepted...
iv = randbytes(16)
mac = cbc_mac(message, iv)

forged_iv = xor(xor(iv, message[:16]), forged_message[:16])

# And the server will accept...
authenticate(forged_message, forged_iv, mac)

True

Now let's tune up that protocol a little bit.

As we now know, you're supposed to use a fixed IV with CBC-MAC, so let's do that.  We'll set ours at 0 for simplicity.  This means the IV comes out of the protocol:

```
message || MAC
```

Pretty simple, but we'll also adjust the message.  For the purposes of efficiency, the bank wants to be able to process multiple transactions in a single request.  So the message now looks like this:

```
from=#{from_id}&tx_list=#{transactions}
```

With the transaction list formatted like:

```
to:amount(;to:amount)*
```

There's still a weakness here: the MAC is vulnerable to length extension attacks.  How?

Well, the output of CBC-MAC is a valid IV for a new message.

"*But we don't control the IV anymore!*"

With sufficient mastery of CBC, we can fake it.

Your mission: capture a valid message from your target user.  Use length extension to add a transaction paying the attacker's account 1M spacebucks.

> **Hint!**
>
> This would be a lot easier if you had full control over the first block of your message, huh?  Maybe you can simulate that.

Food for thought: *How would you modify the protocol to prevent this?*

---

This attack seems even less feasible than the previous one.  It's a variation on [Challenge 50](#Challenge-50:-Hashing-with-CBC-MAC), next, which we explore fully.  We'll just note here that a successful attack would require two things to happen:

- The attacker must compute the CBC-MAC of the extension (by itself) in which the first block has been XOR-ed with the original message's CBC-MAC.  Thus the client would have to be willing to sign, not just a message that it did not originate, but a message that appears to it to be just a string of arbitrary bytes.

- The computation of the original message's CBC-MAC incorporated PKCS#7 padding, and that padding would have to be manually inserted between the original message and the extension in order to make the CBC-MAC computation come out right.  Thus the server would have to accept a request with padding bytes in the middle.

Both those requirements seem unrealistic.

How can length extension attacks be prevented?  Simply by including some kind of end-of-request marker in the syntax.

In [3]:
message = b"from=Bob&tx_list=Dave:5"
mac = cbc_mac(message, iv=zero)

extension = b";Greg:1000000"

# The scenario calls for us to compute the CBC-MAC of the extension in
# which the first block has been XOR-ed with `mac` above, and using
# IV=0, but for brevity we note that that is mathematically equivalent
# to this call.
new_mac = cbc_mac(extension, iv=mac)

forged_message = pad_pkcs7(message) + extension

authenticate(forged_message, iv=zero, mac=new_mac)

True

So it works.  But here's the message the server would have to accept:

In [4]:
print(forged_message)

b'from=Bob&tx_list=Dave:5\t\t\t\t\t\t\t\t\t;Greg:1000000'


## Challenge 50: Hashing with CBC-MAC

Sometimes people try to use CBC-MAC as a hash function.

This is a bad idea.  Matt Green explains:

> To make a long story short: cryptographic hash functions are public functions (i.e., no secret key) that have the property of collision-resistance (it's hard to find two messages with the same hash).  MACs are keyed functions that (typically) provide message unforgeability -- a very different property.  Moreover, they guarantee this only when the key is secret.

Let's try a simple exercise.

Hash functions are often used for code verification.  This snippet of JavaScript (with newline):

```
alert('MZA who was that?');
```

Hashes to *296b8d7cb78a243dda4d0a61d33bbdd1* under CBC-MAC with a key of "*YELLOW SUBMARINE*" and a 0 IV.

Forge a valid snippet of JavaScript that alerts "Ayo, the Wu is back!" and hashes to the same value.  Ensure that it runs in a browser.

---

JavaScript has the convenient property that a [single-line comment](https://262.ecma-international.org/16.0/index.html#sec-comments) (delimited with "//") extends up to the next newline character and can contain *any* characters, in fact any bytes (other than newline) along the way.  Since the CBC-MAC consists of just the last ciphertext block, the idea is that we add to our forged code (regardless what it does, regardless how long it is) a comment that causes the MAC to turn out right.  Referring again to how CBC mode works:

![](cbc-mode.png)

Let $F$ be the forged code with "//" appended and let $C_F$ be the CBC-MAC of $F$, i.e., its last ciphertext block.  Note that in computing $C_F$ the encryption process will have added some PKCS#7 padding bytes; call this padding $P$.  Let $O$ be just the first block (i.e., 16 bytes) of the original code.  Let $O' = C_F \oplus O$.  Consider manually padding $F$ and encrypting $F \parallel P \parallel O'$.  In computing the ciphertext block corresponding to $O'$ we will be encrypting $C_F \oplus O' = C_F \oplus (C_F \oplus O) = O$.  I.e., we will have recreated the start of the original CBC-MAC computation.  From this point, if we append the remainder of the original code (i.e., from the second block on) verbatim we will wind up matching the original CBC-MAC.

There are two potential problems.  One, if 10 PKCS#7 padding bytes are required we will end up appending newline (ASCII 10) bytes to $F$, which will mess up our commenting mechanism.  Two, it may happen that a newline appears in $O'$.  We don't address these problems here, but they could both be handled by making some kind of innocuous change to the forged code.  And there is one limitation: we're assuming the original code consists of a single line, so that its actions can be disabled by a single comment delimiter.  If the original code has multiple lines, we will be commenting out only the first line.  Our approach requires that the original code appear at the end of the plaintext verbatim, so we can't add further comment delimiters.  A potential workaround would be to add some kind of quit or exit call at the end of the forged code, to avoid running the original code, but the success of this approach would seem to depend on the particulars of the original code (e.g., that commenting out the first line does not create a syntax error).

In [5]:
def cbc_mac_hash(text):
    return aes_128_cbc_encrypt(text, b"YELLOW SUBMARINE", zero)[-16:]

original_code = b"alert('MZA who was that?');\n"
print(cbc_mac_hash(original_code).hex())

forged_code = b"alert('Ayo, the Wu is back!');\n" + b"//"
C_f = cbc_mac_hash(forged_code)
forged_code = (
    pad_pkcs7(forged_code) +
    xor(C_f, original_code[:16]) +
    original_code[16:]
)

print(cbc_mac_hash(forged_code).hex())

296b8d7cb78a243dda4d0a61d33bbdd1
296b8d7cb78a243dda4d0a61d33bbdd1


The forged code looks like this:

In [6]:
print(forged_code)

b'alert(\'Ayo, the Wu is back!\');\n//\x0f\x0f\x0f\x0f\x0f\x0f\x0f\x0f\x0f\x0f\x0f\x0f\x0f\x0f\x0f\xe3l\xa7\xc1BR\x0c\xa8Dy"8tA\xaa~as that?\');\n'


## Challenge 51: Compression ratio side-channel attacks

Internet traffic is often compressed to save bandwidth.  Until recently, this included HTTPS headers, and it still includes the contents of responses.

Why does that matter?

Well, if you're an attacker with:

1. Partial plaintext knowledge *and*
2. Partial plaintext control *and*
3. Access to a compression oracle

You've got a pretty good chance to recover any additional unknown plaintext.

What's a compression oracle?  You give it some input and it tells you how well the full message compresses, i.e., the length of the resultant output.

This is somewhat similar to the timing attacks we did way back in [set 4](set-4.html) in that we're taking advantage of incidental side channels rather than attacking the cryptographic mechanisms themselves.

Scenario: you are running a MITM attack with an eye towards stealing secure session cookies.  You've injected malicious content allowing you to spawn arbitrary requests and observe them in flight.  (The particulars aren't terribly important, just roll with it.)

So!  Write this oracle:

```
oracle(P) -> length(encrypt(compress(format_request(P))))
```

Format the request like this:

```
POST / HTTP/1.1
Host: hapless.com
Cookie: sessionid=TmV2ZXIgcmV2ZWFsIHRoZSBXdS1UYW5nIFNlY3JldCE=
Content-Length: ((len(P)))
((P))
```

(Pretend you can't see that session id.  You're the attacker.)

Compress using zlib or whatever.

Encryption... is actually kind of irrelevant for our purposes, but be a sport.  Just use some stream cipher.  Dealer's choice.  Random key/IV on every call to the oracle.

And then just return the length in bytes.

Now, the idea here is to leak information using the compression library.  A payload of "sessionid=T" should compress just a little bit better than, say, "sessionid=S".

There is one complicating factor.  The DEFLATE algorithm operates in terms of individual bits, but the final message length will be in bytes.  Even if you do find a better compression, the difference may not cross a byte boundary.  So that's a problem.

You may also get some incidental false positives.

But don't worry!  I have full confidence in you.

Use the compression oracle to recover the session id.

I'll wait.

Got it?  Great.

---

The session key is "Never reveal the Wu-Tang Secret!" Base64-encoded.

This attack was perhaps first described, abstractly, by John Kelsey in 2002 in [Compression and Information Leakage of Plaintext](https://doi.org/10.1007/3-540-45661-9_21) and then exploited a decade later in the [CRIME](https://en.wikipedia.org/wiki/CRIME) and [BREACH](https://en.wikipedia.org/wiki/BREACH) attacks on web browsers.  The challenge here describes it as a MITM attack, but it's better characterized as one in which the attacker can supply input to a system, have that input reflected in the (compressed) output from that system, and observe the length of the output.  If the system is a web browser, the input could be supplied in the body of a POST request (as suggested here) or in the URL itself (i.e., just visit URL /sessionid=...); the output length could be determined by packet sniffing.  If the system is a web server, then the attack would rely on the server reflecting the input in the response.

The only protection seems to be to disable compression.

Fundamentally, the attack works because the [DEFLATE/LZ77 algorithm](https://zlib.net/feldspar.html) used by zlib will replace a previously-seen substring within a sliding window with a back reference to the earlier occurrence.  Here we're relying on the existence of an anchor of known bytes to provide the match the algorithm is looking for, e.g., `sessionid=`.  We input such an anchor, and if some number of bytes following that anchor match the full and correct session ID already embedded in the message, then fewer bits will be required to compress the subsequent bytes that do not match.  In developing our solution we found that we were able to consistently recover the session ID one byte at a time using anchor `Cookie: sessionid=`, whereas using just anchor `sessionid=` we had to guess two bytes at a time.

The padding argument will be used in the second part of this challenge.

In [7]:
session_id = b"TmV2ZXIgcmV2ZWFsIHRoZSBXdS1UYW5nIFNlY3JldCE="

base64_alphabet = (
    b"ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    b"abcdefghijklmnopqrstuvwxyz"
    b"0123456789/+="
)

def format_request(data):
    header = (
        "POST / HTTP/1.1\r\n"
        "Host: hapless.com\r\n"
        "Cookie: sessionid={session_id}\r\n"
        "Content-Length: {length}\r\n"
        "\r\n"
    )
    return header.format(
        session_id=session_id.decode(),
        length=len(data)
    ).encode() + data

def oracle(data):
    key = randbytes(16)
    nonce = randbytes(16)
    return len(aes_128_ctr_crypt(compress(format_request(data)), key, nonce))

def recover_session_id(add_padding=False):
    id = bytes()
    while not id.endswith(b"\r\n"):
        padding = b""
        while True:
            l = [
                [oracle(padding + b"Cookie: sessionid=" + id + bytes([b])), b]
                for b in base64_alphabet + b"\r\n"
            ]
            l.sort(key=lambda t: t[0])
            if l[0][0] < l[1][0]:
                # There is a unique shortest compressed output
                id += bytes([l[0][1]])
                break
            else:
                assert add_padding
                padding += randbytes(1)
    return id[:-2]

print(recover_session_id() == session_id)

True


Now swap out your stream cipher for CBC and do it again.

---

A block cipher makes this quite a bit harder because the added PKCS#7 padding obscures the length of the nominally compressed output.  To counter this we add arbitrary content (designed to compress poorly, so as to have effect) one byte at a time, until, in testing all possibilities for the next byte of the session ID, there is one nominally compressed output that is shorter than the others, and which manages to make the corresponding encrypted output one block shorter.  In practice this required adding 30-60 bytes.

In [8]:
def oracle(data):
    key = randbytes(16)
    iv = randbytes(16)
    return len(aes_128_cbc_encrypt(compress(format_request(data)), key, iv))

print(recover_session_id(add_padding=True) == session_id)

True
